# Sentiment analysis with a SimpleRNN
Two ways to turn text into numbers: integer encoding, and a learned embedding. First on a toy document,
then on the IMDB movie reviews, with the same SimpleRNN on top.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

tf.config.experimental.enable_op_determinism()   # same numbers on every run
keras.utils.set_random_seed(42)
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## Integer encoding a toy document
`TextVectorization` lower-cases, strips punctuation, splits on spaces, builds the vocabulary
(most frequent word first) and replaces each word by its index. Index 0 is padding, 1 is "[UNK]",
the out-of-vocabulary token.

In [ ]:
docs = ["go india", "india india", "hip hip hurray", "jeetega bhai jeetega india jeetega",
        "bharat mata ki jai", "kohli kohli", "sachin sachin", "dhoni dhoni", "modi ji ki jai",
        "inquilab zindabad"]
vectorizer = keras.layers.TextVectorization(output_mode="int")
vectorizer.adapt(docs)
vocab = vectorizer.get_vocabulary()
print(len(vocab), "entries:", {w: i for i, w in enumerate(vocab)})

In [ ]:
# Word counts (how often each word appears)
from collections import Counter
print(Counter(" ".join(docs).split()).most_common(5))

In [ ]:
# Each sentence becomes a sequence of indices; TextVectorization pads with 0 at the end
sequences = vectorizer(docs).numpy()
print(sequences)
print("unseen word ->", vectorizer(["go pakistan"]).numpy())   # 'pakistan' is not in the vocabulary

In [ ]:
# The same padding step with pad_sequences, for lists of different lengths
ragged = [[vocab.index(w) for w in d.split()] for d in docs]
print(keras.utils.pad_sequences(ragged, padding="post"))

## The Embedding layer on the toy document
Embedding(input_dim = vocabulary size, output_dim = 2): every index becomes 2 numbers.

In [ ]:
emb_model = keras.Sequential([keras.Input(shape=(sequences.shape[1],)), keras.layers.Embedding(len(vocab), 2)])
emb_model.summary()
out = emb_model.predict(sequences, verbose=0)
print("output shape:", out.shape)                       # (sentences, words, 2)
print("first sentence:\n", out[0].round(3))

In [ ]:
# An embedding lookup is a one-hot vector times the weight matrix: it picks one row
E = emb_model.layers[0].get_weights()[0]                # shape (vocabulary size, 2)
idx = sequences[0, 0]
one_hot = np.eye(len(vocab))[idx]
print("row", idx, "of E:", E[idx].round(4), " one-hot @ E:", (one_hot @ E).round(4))
assert np.allclose(one_hot @ E, E[idx]) and np.allclose(out[0, 0], E[idx])

## IMDB: already integer encoded

In [ ]:
(X_train_full, y_train), (X_test_full, y_test) = keras.datasets.imdb.load_data()
print(len(X_train_full), "training reviews,", len(X_test_full), "test reviews")
print("first review starts:", X_train_full[0][:10])
print("lengths of the first three reviews:", [len(x) for x in X_train_full[:3]])
print("median length:", int(np.median([len(x) for x in X_train_full])))

In [ ]:
word_index = keras.datasets.imdb.get_word_index()
index_word = {i + 3: w for w, i in word_index.items()}   # load_data shifts every index by 3
index_word.update({0: "<pad>", 1: "<start>", 2: "<unk>"})
print(" ".join(index_word[i] for i in X_train_full[0][:20]))

In [ ]:
# Keep 50 words per review. pad_sequences pads and truncates at the start by default ("pre"),
# so a long review keeps its LAST 50 words.
MAXLEN = 50
X_train = keras.utils.pad_sequences(X_train_full, maxlen=MAXLEN)
X_test = keras.utils.pad_sequences(X_test_full, maxlen=MAXLEN)
print(X_train.shape, " last 5 kept:", X_train[0][-5:], " original last 5:", X_train_full[0][-5:])

## Approach 1: integer encoding into a SimpleRNN
Input shape (50, 1): 50 time steps, 1 number per time step.

In [ ]:
def integer_model():
    return keras.Sequential([keras.Input(shape=(MAXLEN, 1)),
                             keras.layers.SimpleRNN(32, return_sequences=False),
                             keras.layers.Dense(1, activation="sigmoid")])


integer_model().summary()

## Approach 2: a learned embedding into the same SimpleRNN
Only the 10,000 most frequent words; rarer words become index 2 ("<unk>").

In [ ]:
VOCAB = 10_000
(Xe_train, _), (Xe_test, _) = keras.datasets.imdb.load_data(num_words=VOCAB)
Xe_train = keras.utils.pad_sequences(Xe_train, maxlen=MAXLEN)
Xe_test = keras.utils.pad_sequences(Xe_test, maxlen=MAXLEN)


def embedding_model():
    return keras.Sequential([keras.Input(shape=(MAXLEN,)),
                             keras.layers.Embedding(VOCAB, 2),
                             keras.layers.SimpleRNN(32, return_sequences=False),
                             keras.layers.Dense(1, activation="sigmoid")])


embedding_model().summary()

## Train both, 3 seeds each, 5 epochs, the test set as validation data

In [ ]:
rows, models = [], {}
for name, build, (Xtr, Xte) in (("integer encoding", integer_model, (X_train[..., None].astype("float32"), X_test[..., None].astype("float32"))),
                                 ("embedding", embedding_model, (Xe_train, Xe_test))):
    for seed in range(3):
        keras.utils.set_random_seed(seed)
        model = build()
        model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
        h = model.fit(Xtr, y_train, epochs=5, batch_size=32, validation_data=(Xte, y_test), verbose=0)
        for e in range(5):
            rows.append(dict(approach=name, seed=seed, epoch=e + 1, accuracy=h.history["accuracy"][e],
                             val_accuracy=h.history["val_accuracy"][e]))
        models[name, seed] = model
        print(f"{name:17} seed {seed}: train {h.history['accuracy'][-1]:.3f}  test {h.history['val_accuracy'][-1]:.3f}")
curves = pd.DataFrame(rows)
curves.round(4).to_csv(DATA / "imdb_curves.csv", index=False)
print(curves.groupby(["approach", "epoch"])[["accuracy", "val_accuracy"]].mean().round(3))

## What the embedding learned
Each word is now 2 numbers. Words that carry sentiment should sit apart.

In [ ]:
E = models["embedding", 0].layers[0].get_weights()[0]
groups = {"positive": ["great", "excellent", "wonderful", "best", "perfect", "loved", "brilliant", "amazing"],
          "negative": ["worst", "awful", "waste", "boring", "terrible", "bad", "poor", "horrible"],
          "neutral": ["the", "and", "movie", "film", "actor", "story", "was", "of"]}
emb_rows = [dict(word=w, group=g, e1=E[word_index[w] + 3, 0], e2=E[word_index[w] + 3, 1])
            for g, ws in groups.items() for w in ws]
emb = pd.DataFrame(emb_rows)
emb.round(4).to_csv(DATA / "embedding_words.csv", index=False)
print(emb.groupby("group")[["e1", "e2"]].mean().round(3))